<a href="https://colab.research.google.com/github/Aanjneya007/NLP/blob/main/Sentiment_Analysis_BERT_CS2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Case Study: Sentiment Analysis using BERT

In this notebook, we will develop a sentiment analysis program using the state-of-the-art **BERT (Bidirectional Encoder Representations from Transformers)** model. We'll use the Hugging Face `transformers` library to make it easy to load, tokenize, and run predictions on text.

In [1]:
# Install necessary libraries
!pip install -q transformers torch

We will load the pre-trained `distilbert-base-uncased-finetuned-sst-2-english` model. This is a lightweight version of BERT fine-tuned on the Stanford Sentiment Treebank (SST-2) dataset, which is ideal for binary sentiment classification (positive/negative).

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

# Define the model name
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

# Load pre-trained tokenizer and model
print("Loading BERT model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
print("Model loaded successfully!")

Loading BERT model and tokenizer...


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded successfully!


Now, let's build a clean, reusable function to analyze the sentiment of any given text input.

In [3]:
def analyze_sentiment(text):
    # Tokenize input text and convert to tensor
    inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt")

    # Run inference without tracking gradients
    with torch.no_grad():
        outputs = model(**inputs)

    # Apply softmax to output logits to calculate confidence probabilities
    probabilities = F.softmax(outputs.logits, dim=-1)

    # Get the index of the highest probability
    prediction = torch.argmax(probabilities, dim=-1).item()
    confidence = probabilities[0][prediction].item()

    # Map model labels (0: NEGATIVE, 1: POSITIVE) to text
    sentiment = "POSITIVE" if prediction == 1 else "NEGATIVE"

    return sentiment, confidence

Let's test our sentiment analysis pipeline on various sample sentences to see how it performs!

In [4]:
test_sentences = [
    "I absolutely loved this movie! The acting was incredible and the plot was engaging.",
    "The food at the restaurant was mediocre and the service was extremely slow.",
    "It was an okay experience, nothing special but not terrible either.",
    "This is the best purchase I have ever made!"
]

for sentence in test_sentences:
    sentiment, confidence = analyze_sentiment(sentence)
    print(f"Text: {sentence}")
    print(f"Sentiment: {sentiment} ({confidence*100:.2f}% confidence)\n" + "-"*50)

Text: I absolutely loved this movie! The acting was incredible and the plot was engaging.
Sentiment: POSITIVE (99.99% confidence)
--------------------------------------------------
Text: The food at the restaurant was mediocre and the service was extremely slow.
Sentiment: NEGATIVE (99.98% confidence)
--------------------------------------------------
Text: It was an okay experience, nothing special but not terrible either.
Sentiment: POSITIVE (80.28% confidence)
--------------------------------------------------
Text: This is the best purchase I have ever made!
Sentiment: POSITIVE (99.99% confidence)
--------------------------------------------------
